# A session from LOBSTER files

`MarketSession` has four constructors and this is the one that folds nothing: the orderbook
file already *is* the sequence of book states, and the message file supplies the clock and
the trades.

The reference for the format and for every stage of the pipeline is
[`documentation/from-lobster-files-to-a-session.md`](../documentation/from-lobster-files-to-a-session.md).
This notebook is the part that needs the data, which the repository does not ship: what a
windowed read costs, what the lit tape leaves out, what truncation does to the coverage
flags, and how far LOBSTER's idea of a row is from ours.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from unito26.lob import frames, lobster
from unito26.lob.messages import GridDepth, SweepSize, TickGrid
from unito26.lob.session import MarketSession
from unito26.lob.statistics import SessionStatistics

DATA = Path("..") / "data" / "lobster"
CENT = TickGrid(0.01)
SPEC = SessionStatistics((GridDepth(1), GridDepth(5)), (SweepSize(100),), (1, 300))

## 1. The pair names its own schema

`LEVEL` is in the filename, so the number of columns and their names are fixed before a byte
is read. That is the whole argument for parsing the name rather than passing two paths: the
paths are the same type and silently swappable, and a depth-10 file read as depth 5 is not an
error at all — only a different statistic under the same name.

In [2]:
files = lobster.LobsterFiles.parse(
    DATA / "AMZN_2012-06-21_34200000_57600000_message_10.csv"
)
print(files.ticker, files.day, "depth", files.reported_depth)
print("span from the filename (ms -> s):", files.span)
print("orderbook twin:", files.orderbook_path.name)
print("columns known before reading:", len(frames.lobster_book_columns(files.reported_depth)))

AMZN 2012-06-21 depth 10
span from the filename (ms -> s): TradingWindow(opens=34200.0, closes=57600.0)
orderbook twin: AMZN_2012-06-21_34200000_57600000_orderbook_10.csv
columns known before reading: 40


## 2. What a window costs

The message file is the smaller of the pair and carries the only clock, so it is read whole
and used as the index into the other. Only the rows the window covers are parsed. The
saving is not subtle: SPY at depth 50 is a 1.4 GB orderbook file.

In [3]:
def cost(files, window, label):
    import time
    started = time.perf_counter()
    messages, book = lobster.load_aligned(files, window)
    seconds = time.perf_counter() - started
    return {
        "case": label,
        "rows": len(book),
        "of": lobster._count_rows(files.orderbook_path),
        "seconds": round(seconds, 2),
        "book MB": round(book.memory_usage(deep=True).sum() / 1e6, 1),
    }


spy = lobster.LobsterFiles.parse(DATA / "SPY_2012-06-21_34200000_37800000_message_50.csv")
five_minutes = lobster.TradingWindow(35400.0, 35700.0)

pd.DataFrame([
    cost(files, lobster.NASDAQ_REGULAR_HOURS, "AMZN d10, whole session"),
    cost(files, five_minutes, "AMZN d10, five minutes"),
    cost(spy, five_minutes, "SPY d50, five minutes"),
]).set_index("case")

,rows,of,seconds,book MB
case,,,,
"AMZN d10, whole session",269748,269748,0.95,86.3
"AMZN d10, five minutes",4005,269748,0.40,1.3
"SPY d50, five minutes",77829,1154737,3.80,124.5


The bound is set by the rows kept, not by where they sit in the file: skipping still streams
the text, so a window at the end costs the same memory and more time than one at the start.

## 3. The session, and what the lit tape leaves out

`trades` counts **visible** executions only. A type-5 print is a trade against an order that
was never displayed; it rests at or inside the touch, so it prints better than the visible
tape. Counting it would give the traded average of a tape the book never showed.

In [4]:
session = MarketSession.from_lobster_files(
    files, SPEC, CENT, lobster.NASDAQ_REGULAR_HOURS, True
)
print("rows:", len(session.lobster_book), " truncated:", session.truncated,
      " price unit:", session.price_unit)

messages, book = lobster.load_aligned(files, lobster.NASDAQ_REGULAR_HOURS)
unit = lobster.price_unit(CENT)
kinds = {"visible (type 4)": 4, "hidden (type 5)": 5}
rows = []
for label, kind in kinds.items():
    taken = messages["Type"] == kind
    volume = messages.loc[taken, "Size"].to_numpy()
    price = messages.loc[taken, "Price"].to_numpy() / unit
    rows.append({"tape": label, "prints": int(taken.sum()), "shares": int(volume.sum()),
                 "VWAP (ticks)": round(float((volume * price).sum() / volume.sum()), 2)})
both = messages["Type"].isin([4, 5])
volume = messages.loc[both, "Size"].to_numpy()
price = messages.loc[both, "Price"].to_numpy() / unit
rows.append({"tape": "both", "prints": int(both.sum()), "shares": int(volume.sum()),
             "VWAP (ticks)": round(float((volume * price).sum() / volume.sum()), 2)})
pd.DataFrame(rows).set_index("tape")

rows: 269748  truncated: True  price unit: 100


,prints,shares,VWAP (ticks)
tape,,,
visible (type 4),8974,613248,22264.01
hidden (type 5),2445,197507,22261.81
both,11419,810755,22263.48


In [5]:
# Where the hidden prints sit relative to the touch that stood before them.
mid = (book["AskPrice1"] + book["BidPrice1"]).to_numpy() / 2 / unit
before = np.concatenate(([np.nan], mid[:-1]))
spread = (book["AskPrice1"] - book["BidPrice1"]).to_numpy() / unit
prior_half = np.concatenate(([np.nan], spread[:-1] / 2))
price = messages["Price"].to_numpy() / unit

inside = np.abs(price - before) < prior_half
for label, kind in kinds.items():
    taken = (messages["Type"] == kind).to_numpy() & np.isfinite(before)
    share = float((messages["Size"].to_numpy()[taken & inside]).sum()
                  / messages["Size"].to_numpy()[taken].sum())
    print(f"{label:>18}: {share:6.1%} of volume prints strictly inside the prior spread")

  visible (type 4):   0.0% of volume prints strictly inside the prior spread
   hidden (type 5): 100.0% of volume prints strictly inside the prior spread


## 4. Truncation, and a coverage flag that weakens

A padded level in a *file* does not mean the side ended. It means nothing further was seen
inside the visible price range, which is a weaker claim than a book we folded makes — and it
is why the session carries `truncated`. Padding is an opening artefact: at depth 50 the book
has not yet filled to fifty levels.

In [6]:
aapl = lobster.LobsterFiles.parse(DATA / "AAPL_2012-06-21_34200000_37800000_message_50.csv")
opening = lobster.TradingWindow(34200.0, 34500.0)
deep = MarketSession.from_lobster_files(aapl, SPEC, CENT, opening, True)

padded = (deep.lobster_book["AskPrice50"] == frames.ASK_PADDING).to_numpy()
print(f"{padded.sum()} of {len(padded)} rows pad the fiftieth ask level")
print("all within the first", round(deep.lobster_book.index[padded].max() - 34200.0, 2), "seconds")
levels = sorted(deep.stats.loc[padded, "AskOccupiedLevels"].astype(int).unique().tolist())
print("occupied ask levels there:", levels[:6], "...", levels[-1])

108 of 8812 rows pad the fiftieth ask level
all within the first 0.64 seconds
occupied ask levels there: [40, 41, 42, 43, 44, 45] ... 49


## 5. One order, several rows

LOBSTER records the execution of each *resting* order, not the trade. An incoming order that
consumes five resting ones writes five rows sharing one timestamp, and the intermediate book
states between them are configurations that exist inside the matching of a single order.

Grouped on `TimeNanoseconds` — the exact integer key, never the parsed float.

In [7]:
def granularity(name):
    pair = lobster.LobsterFiles.parse(DATA / name)
    messages, _ = lobster.load_aligned(pair, lobster.NASDAQ_REGULAR_HOURS)
    lit = messages[messages["Type"] == 4]
    key = lit["TimeNanoseconds"].to_numpy()
    side = lit["Direction"].to_numpy()
    price = lit["Price"].to_numpy()
    fresh = np.concatenate(([True], (key[1:] != key[:-1]) | (side[1:] != side[:-1])))
    trade = np.cumsum(fresh) - 1
    sizes = np.bincount(trade)
    prices = pd.Series(price).groupby(trade).nunique().to_numpy()
    multi = sizes > 1
    return {
        "ticker": pair.ticker,
        "executions": len(lit),
        "trades": len(sizes),
        "multi-row": int(multi.sum()),
        "rows inside": f"{sizes[multi].sum() / len(lit):.0%}",
        "largest": int(sizes.max()),
        "queue splits": f"{(prices[multi] == 1).sum() / multi.sum():.0%}",
    }


pd.DataFrame([
    granularity("AMZN_2012-06-21_34200000_57600000_message_10.csv"),
    granularity("GOOG_2012-06-21_34200000_57600000_message_10.csv"),
    granularity("AAPL_2012-06-21_34200000_57600000_message_10.csv"),
    granularity("INTC_2012-06-21_34200000_57600000_message_10.csv"),
]).set_index("ticker")

,executions,trades,multi-row,rows inside,largest,queue splits
ticker,,,,,,
AMZN,8974,6591,1511,43%,28,80%
GOOG,7765,5931,1190,39%,35,79%
AAPL,23658,18016,3883,40%,43,82%
INTC,28924,8006,3888,86%,105,99%


The last column is the one that matters. A multi-row trade is either a *level walk*, which
an aggregate book could report one level at a time if it chose, or a *queue split* — several
resting orders at one price, which no sequence of aggregate states determines, because the
aggregate book does not know a level of 100 is five orders of 20.

Four fifths of them are queue splits. So a folded session and a loaded one carry the same
columns, the same dtypes and the same index name, and a different unit of observation; no
schema can say so. What to do about it is open, in
[`dev-context/lobster-execution-granularity.md`](../dev-context/lobster-execution-granularity.md).